# Preprocessing and Feature Engineering

### Exhibition of Datasets



In [ ]:
# Libraries
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import nltk
import string
import re
from datetime import datetime
from collections import Counter
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.stem.porter import PorterStemmer
from nltk.tokenize import word_tokenize
from nltk.tokenize import sent_tokenize

In [ ]:
# yFinance Apple Historical Data

ticker_symbol = "AAPL"     # ticker symbol

ticker = yf.Ticker(ticker_symbol)   # ticker object

# Fetching historical data from 2016-01-01 to 2024-12-31
historical_data = ticker.history(start="2016-01-01", end="2024-12-31")

# Save historical data to CSV
historical_data[['Open', 'High', 'Low', 'Close', 'Volume']].to_csv('aapl_historical_data.csv')

print(historical_data[['Open', 'High', 'Low', 'Close', 'Volume']])

                                 Open        High         Low       Close  \
Date                                                                        
2016-01-04 00:00:00-05:00   23.184077   23.807681   23.046251   23.803162   
2016-01-05 00:00:00-05:00   23.893534   23.916128   23.138883   23.206665   
2016-01-06 00:00:00-05:00   22.720890   23.129849   22.564990   22.752522   
2016-01-07 00:00:00-05:00   22.296120   22.623738   21.787747   21.792265   
2016-01-08 00:00:00-05:00   22.266747   22.393275   21.862307   21.907495   
...                               ...         ...         ...         ...   
2024-12-23 00:00:00-05:00  254.156919  255.034791  252.840088  254.655716   
2024-12-24 00:00:00-05:00  254.875189  257.588630  254.675658  257.578674   
2024-12-26 00:00:00-05:00  257.568678  259.474086  257.010028  258.396667   
2024-12-27 00:00:00-05:00  257.209530  258.077462  252.451019  254.974930   
2024-12-30 00:00:00-05:00  251.623005  252.889953  250.146571  251.593079   

In [ ]:
# Apple Daily News Dataset

import pandas as pd
news_df = pd.read_csv("/content/apple_news_data.csv")
print(news_df.head())

                        date  \
0  2024-11-27T16:39:00+00:00   
1  2024-11-26T00:00:00+00:00   
2  2024-11-26T00:00:00+00:00   
3  2024-11-26T00:00:00+00:00   
4  2024-11-26T00:00:00+00:00   

                                               title  \
0  Berkshire Stock Hits Record Even as Company Re...   
1                      What Is a Stock Market Index?   
2  Could Investing $1,000 in Apple Make You a Mil...   
3                       Dow Jones Industrial Average   
4                         What Is the S&P 500 Index?   

                                             content  \
0  Warren Buffett’s caution, his advancing age, a...   
1                      What Is a Stock Market Index?   
2  Could Investing $1,000 in Apple Make You a Mil...   
3                       Dow Jones Industrial Average   
4                         What Is the S&P 500 Index?   

                                                link  \
0  https://finance.yahoo.com/m/f5df3aa4-364b-31d6...   
1  https://www.fool.c

### Data Preprocessing

To prepare the data for modeling, we performed initial cleaning by removing null values, dropping unused columns (e.g., tags, headline link, and pre-calculated sentiment scores), merge datasets, correcting date formats, and standardizing the text by lowercasing all news content. This ensures consistency and high-quality input for both time-series and NLP-based modeling steps.

In [ ]:
# looking at all features
print("News Data: " + str(news_df.columns.tolist()))
print("YFinance Data: " + str(historical_data.columns.tolist()))

News Data: ['date', 'title', 'content', 'link', 'symbols', 'tags', 'sentiment_polarity', 'sentiment_neg', 'sentiment_neu', 'sentiment_pos']
YFinance Data: ['Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits']


In [ ]:
print(news_df.info())
print('\n')
print(historical_data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29752 entries, 0 to 29751
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   date                29752 non-null  object 
 1   title               29752 non-null  object 
 2   content             29752 non-null  object 
 3   link                29752 non-null  object 
 4   symbols             29752 non-null  object 
 5   tags                17564 non-null  object 
 6   sentiment_polarity  29737 non-null  float64
 7   sentiment_neg       29737 non-null  float64
 8   sentiment_neu       29737 non-null  float64
 9   sentiment_pos       29737 non-null  float64
dtypes: float64(4), object(6)
memory usage: 2.3+ MB
None


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2263 entries, 2016-01-04 00:00:00-05:00 to 2024-12-30 00:00:00-05:00
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0  

In [ ]:
# checking to see null values per column
print("News data: ")
news_df.isna().sum()

News data: 


,0
date,0
title,0
content,0
link,0
symbols,0
tags,12188
sentiment_polarity,15
sentiment_neg,15
sentiment_neu,15
sentiment_pos,15


In [ ]:
# checking to see null values per column
print("yfinance data: ")
historical_data.isna().sum()

yfinance data: 


,0
Open,0
High,0
Low,0
Close,0
Volume,0
Dividends,0
Stock Splits,0


In [ ]:
# dropping sentiment analysis and tags (40% values are null)
news_df = news_df.drop(['sentiment_polarity', 'sentiment_neg', 'sentiment_neu', 'sentiment_pos', 'tags','link'], axis = 1)

In [ ]:
# formatting date to remove the additional timestamp in news_df
news_df['date'] = pd.to_datetime(news_df['date'], utc=True).dt.strftime('%m-%d-%Y')

In [ ]:
# formatting the date column in historical data
historical_data = historical_data.reset_index()
historical_data = historical_data.rename(columns={'index': 'date'})
print(historical_data.columns)
print(historical_data.head())

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends',
       'Stock Splits'],
      dtype='object')
                       Date       Open       High        Low      Close  \
0 2016-01-04 00:00:00-05:00  23.184077  23.807681  23.046251  23.803162   
1 2016-01-05 00:00:00-05:00  23.893534  23.916128  23.138883  23.206665   
2 2016-01-06 00:00:00-05:00  22.720890  23.129849  22.564990  22.752522   
3 2016-01-07 00:00:00-05:00  22.296120  22.623738  21.787747  21.792265   
4 2016-01-08 00:00:00-05:00  22.266747  22.393275  21.862307  21.907495   

      Volume  Dividends  Stock Splits  
0  270597600        0.0           0.0  
1  223164000        0.0           0.0  
2  273829600        0.0           0.0  
3  324377600        0.0           0.0  
4  283192000        0.0           0.0  


In [ ]:
historical_data = historical_data.rename(columns={'Date': 'date'})
historical_data = historical_data.drop(columns=['Dividends', 'Stock Splits'])

In [ ]:
news_df['date'] = pd.to_datetime(news_df['date'])
historical_data['date'] = pd.to_datetime(historical_data['date'])

In [ ]:
news_df['date'] = news_df['date'].dt.tz_localize(None)
historical_data['date'] = historical_data['date'].dt.tz_localize(None)

Preprocessing Headlines

In [ ]:
# text preprocessing
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

# Lowercase text columns
news_df['title'] = news_df['title'].str.lower()
news_df['content'] = news_df['content'].str.lower()

# Tokenization
news_df['tokenized_title'] = news_df['title'].apply(word_tokenize)
news_df['tokenized_content'] = news_df['content'].apply(word_tokenize)

# Stop word removal
stop_words = set(stopwords.words("english"))
news_df['tokenized_title'] = news_df['tokenized_title'].apply(lambda tokens: [word for word in tokens if word not in stop_words])
news_df['tokenized_content'] = news_df['tokenized_content'].apply(lambda tokens: [word for word in tokens if word not in stop_words])

# Punctuation removal
news_df['tokenized_title'] = news_df['tokenized_title'].apply(lambda tokens: [word for word in tokens if word.isalnum()])
news_df['tokenized_content'] = news_df['tokenized_content'].apply(lambda tokens: [word for word in tokens if word.isalnum()])

# Removing numbers
news_df['tokenized_title'] = news_df['tokenized_title'].apply(lambda tokens: [word for word in tokens if not word.isdigit()])
news_df['tokenized_content'] = news_df['tokenized_content'].apply(lambda tokens: [word for word in tokens if not word.isdigit()])

# Lemmatization (keeping it separate to see the differne between lemmatized and non-lemmatized words)
lemmatizer = WordNetLemmatizer()
news_df['tokenized_title'] = news_df['tokenized_title'].apply(lambda tokens: [lemmatizer.lemmatize(word) for word in tokens])
news_df['tokenized_content'] = news_df['tokenized_content'].apply(lambda tokens: [lemmatizer.lemmatize(word) for word in tokens])

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


finBERT using HuggingFace and PyTorch for Sentiment Analysis

In [ ]:
!pip install transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 73.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification, pipeline

#creating a pipeline to call finbert
finbert = BertForSequenceClassification.from_pretrained("yiyanghkust/finbert-tone", num_labels = 3)
tokenizer = BertTokenizer.from_pretrained("yiyanghkust/finbert-tone")
nlp = pipeline("sentiment-analysis", model = finbert, tokenizer = tokenizer)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

Device set to use cpu


In [ ]:
# creating batches of 64 since its ~30k rows
def batch_texts(text_list, batch_size):
  final_list = []
  for i in range(0, len(text_list), batch_size):
      yield text_list[i:i+batch_size]

# texts is created by joining the tokens back into a string
texts = news_df['tokenized_title'].apply(lambda x: ' '.join(x)).tolist()
results = []
for i, batch in enumerate(batch_texts(texts, 64)):
    results.extend(nlp(batch))

    if i % 10 == 0:
        print(f"Processed {i * 64} headlines...")

labels = [res['label'] for res in results]
scores = [res['score'] for res in results]
news_df['sentiment'] = labels
news_df['sentiment_score'] = scores

Processed 0 headlines...
Processed 640 headlines...
Processed 1280 headlines...
Processed 1920 headlines...
Processed 2560 headlines...
Processed 3200 headlines...
Processed 3840 headlines...
Processed 4480 headlines...
Processed 5120 headlines...
Processed 5760 headlines...
Processed 6400 headlines...
Processed 7040 headlines...
Processed 7680 headlines...
Processed 8320 headlines...
Processed 8960 headlines...
Processed 9600 headlines...
Processed 10240 headlines...
Processed 10880 headlines...
Processed 11520 headlines...
Processed 12160 headlines...
Processed 12800 headlines...
Processed 13440 headlines...
Processed 14080 headlines...
Processed 14720 headlines...
Processed 15360 headlines...
Processed 16000 headlines...
Processed 16640 headlines...
Processed 17280 headlines...
Processed 17920 headlines...
Processed 18560 headlines...
Processed 19200 headlines...
Processed 19840 headlines...
Processed 20480 headlines...
Processed 21120 headlines...
Processed 21760 headlines...
Proce

In [ ]:
news_df.to_csv("news_with_sentiment.csv", index=False)

In [ ]:
news_sentiment_df = pd.read_csv("news_with_sentiment.csv")
news_sentiment_df.head()

,date,title,content,symbols,tokenized_title,tokenized_content,sentiment,sentiment_score
0,2024-11-27,berkshire stock hits record even as company re...,"warren buffett’s caution, his advancing age, a...","0R2V.IL, AAPL.BA, AAPL.MX, AAPL.NEO, AAPL.SN, ...","['berkshire', 'stock', 'hit', 'record', 'even'...","['warren', 'buffett', 'caution', 'advancing', ...",Negative,0.577534
1,2024-11-26,what is a stock market index?,what is a stock market index?,"AAPL.US, AMZN.US, MSFT.US","['stock', 'market', 'index']","['stock', 'market', 'index']",Neutral,0.999201
2,2024-11-26,"could investing $1,000 in apple make you a mil...","could investing $1,000 in apple make you a mil...",AAPL.US,"['could', 'investing', 'apple', 'make', 'milli...","['could', 'investing', 'apple', 'make', 'milli...",Neutral,0.999849
3,2024-11-26,dow jones industrial average,dow jones industrial average,"AAPL.US, AMGN.US, AMZN.US, CSCO.US, GOOG.US, G...","['dow', 'jones', 'industrial', 'average']","['dow', 'jones', 'industrial', 'average']",Neutral,0.999982
4,2024-11-26,what is the s&p 500 index?,what is the s&p 500 index?,"AAPL.US, AMZN.US, GOOG.US, GOOGL.US, META.US, ...","['p', 'index']","['p', 'index']",Neutral,0.999938


Creating Technical Indicators for Historical Data (Feature Engineering)

In [ ]:
# Moving averages
# SMA (Simple Moving Average)
historical_data['SMA_20'] = historical_data['Close'].rolling(window=20).mean()
historical_data['SMA_5'] = historical_data['Close'].rolling(window=5).mean()

# Difference to capture any trend shifts
sma_diff = historical_data['SMA_20'] - historical_data['SMA_5']
historical_data['SMA_diff'] = sma_diff

# EMA (Exponential Moving Average) -  more responsive to recent market changes — useful for short-term trading signals
# Using 12 and 26 for MACD later
historical_data['EMA'] = historical_data['Close'].ewm(span=12, adjust=False).mean()
historical_data['EMA'] = historical_data['Close'].ewm(span=26, adjust=False).mean()

In [ ]:
# RSI (Relative Strength Index)  -  helps identify overbought or oversold conditions
def rsi(prices, period = 14):
  delta = prices.diff()          # calculating teh daily changes
  gain = delta.where(delta > 0,0)
  loss = -delta.where(delta < 0,0)
  avg_gain = gain.rolling(window = period).mean()
  avg_loss = loss.rolling(window = period).mean()
  rs = avg_gain / avg_loss
  rsi = 100.0 - (100.0 / (1.0 + rs))
  return rsi

historical_data['RSI'] = rsi(historical_data['Close'])

In [ ]:
# MACD (Moving Average Convergence Divergence)

macd_line = historical_data['EMA_12'] - historical_data['EMA_26']
signal_line = macd_line.ewm(span=9, adjust=False).mean()
historical_data['MACD'] = macd_line - signal_line

In [ ]:
# lag sentiment - yesterday's headline could make changes in today's stock
news_sentiment_df['sentiment_score_lag'] = news_sentiment_df['sentiment_score'].shift(1)

Merging datasets

In [ ]:
#merge the two datasets
news_sentiment_df['date'] = pd.to_datetime(news_sentiment_df['date'])

merged_df = pd.merge(news_sentiment_df, historical_data, on='date', how='inner')
print(merged_df.shape)

#save csv
merged_df.to_csv('merged_data.csv', index=False)

(26581, 18)
        date                                              title  \
0 2024-11-27  berkshire stock hits record even as company re...   
1 2024-11-26                      what is a stock market index?   
2 2024-11-26  could investing $1,000 in apple make you a mil...   
3 2024-11-26                       dow jones industrial average   
4 2024-11-26                         what is the s&p 500 index?   

                                             content  \
0  warren buffett’s caution, his advancing age, a...   
1                      what is a stock market index?   
2  could investing $1,000 in apple make you a mil...   
3                       dow jones industrial average   
4                         what is the s&p 500 index?   

                                             symbols  \
0  0R2V.IL, AAPL.BA, AAPL.MX, AAPL.NEO, AAPL.SN, ...   
1                          AAPL.US, AMZN.US, MSFT.US   
2                                            AAPL.US   
3  AAPL.US, AMGN.US, AMZ

In [ ]:
print("Merged Data: " + str(merged_df.columns.tolist()))

Merged Data: ['date', 'title', 'content', 'symbols', 'tokenized_title', 'tokenized_content', 'sentiment', 'sentiment_score', 'Open', 'High', 'Low', 'Close', 'Volume', 'SMA_20', 'SMA_5', 'SMA_diff', 'EMA', 'RSI']


In [ ]:
# looking at the top 5 rows of news_df
print(merged_df.head(5))

        date                                              title  \
0 2024-11-27  berkshire stock hits record even as company re...   
1 2024-11-26                      what is a stock market index?   
2 2024-11-26  could investing $1,000 in apple make you a mil...   
3 2024-11-26                       dow jones industrial average   
4 2024-11-26                         what is the s&p 500 index?   

                                             content  \
0  warren buffett’s caution, his advancing age, a...   
1                      what is a stock market index?   
2  could investing $1,000 in apple make you a mil...   
3                       dow jones industrial average   
4                         what is the s&p 500 index?   

                                             symbols  \
0  0R2V.IL, AAPL.BA, AAPL.MX, AAPL.NEO, AAPL.SN, ...   
1                          AAPL.US, AMZN.US, MSFT.US   
2                                            AAPL.US   
3  AAPL.US, AMGN.US, AMZN.US, CSCO.U